In [1]:
from pyod.models.auto_encoder import AutoEncoder
from sklearn.preprocessing import MinMaxScaler
from pyod.utils.data import generate_data
import tensorflow as tf

import pandas as pd
import numpy as np

import os
import pickle

from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt 
%matplotlib inline

import sys
sys.path.append('..')
sys.path.append('../..')

from src.utils import train_test_anomaly, raw_thresholds

2024-04-04 11:44:06.055713: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-04 11:44:06.055755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-04 11:44:06.057360: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-04 11:44:06.065236: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-04 11:44:07.103885: W tensorflow/compiler/tf2

In [2]:
file_path = '../../datasets/Dodgers/101-freeway-traffic.test.out'

columns = ['value', 'anomaly']

df = pd.read_csv(file_path, names=columns, header=None)

In [3]:
df = df[df['value'] >= 0]
df['anomaly'].value_counts()

anomaly
0    44788
1     2709
Name: count, dtype: int64

In [4]:
X_train, _ = generate_data(n_train=200, train_only=True)
X_train

array([[ 2.24605631e+00, -5.60391913e-01],
       [ 1.62118582e+00, -1.71987565e+00],
       [ 1.36247107e+00,  4.36138723e-01],
       [-8.88179324e-01, -1.50334185e+00],
       [ 1.04261706e+00,  3.75244721e-01],
       [ 7.31555840e-01, -5.93844224e-02],
       [-9.97241329e-02, -7.53516226e-01],
       [-7.81123662e-01,  3.33880833e-01],
       [ 1.03513045e+00, -1.10487540e+00],
       [-8.37394638e-01,  3.61797415e-01],
       [ 4.99803164e-01, -4.04262289e-01],
       [-2.51875710e-01, -1.27444799e-01],
       [-1.17805368e+00, -2.09487052e+00],
       [-4.41079158e-01, -9.21134741e-01],
       [ 1.82588091e-01, -3.39926991e-01],
       [ 3.45248042e-01,  7.09498433e-02],
       [ 1.41013680e+00, -2.06321277e+00],
       [ 2.55488567e-01, -2.03116662e+00],
       [-7.93606305e-01, -6.65993259e-01],
       [-6.12590668e-01, -2.07505200e+00],
       [ 4.27429097e-01,  1.73715342e+00],
       [ 1.12296409e+00,  8.31260607e-01],
       [ 2.80404646e+00,  6.84157081e-01],
       [ 1.

In [5]:
train_data, test_data = train_test_anomaly(data=df, shuffle=False)

In [6]:
train_np = train_data[['value']].values.reshape(-1, 1)
train_np

array([[23],
       [42],
       [37],
       ...,
       [46],
       [35],
       [35]])

In [7]:
train_tf = tf.expand_dims(tf.convert_to_tensor(train_np, dtype=tf.float32), axis=0)
train_tf

<tf.Tensor: shape=(1, 33247, 1), dtype=float32, numpy=
array([[[23.],
        [42.],
        [37.],
        ...,
        [46.],
        [35.],
        [35.]]], dtype=float32)>

In [8]:
autoencode = AutoEncoder(hidden_neurons=[1, 32, 8, 8, 32, 1] ,epochs=40, batch_size=64, contamination=0.057, preprocessing=True, random_state=42)

In [9]:
autoencode.fit(train_np)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
 dropout (Dropout)           (None, 1)                 0         
                                                                 
 dense_1 (Dense)             (None, 1)                 2         
                                                                 
 dropout_1 (Dropout)         (None, 1)                 0         
                                                                 
 dense_2 (Dense)             (None, 1)                 2         
                                                                 
 dropout_2 (Dropout)         (None, 1)                 0         
                                                                 
 dense_3 (Dense)             (None, 32)                6

AutoEncoder(batch_size=64, contamination=0.057, dropout_rate=0.2, epochs=40,
      hidden_activation='relu', hidden_neurons=[1, 32, 8, 8, 32, 1],
      l2_regularizer=0.1,
      loss=<function mean_squared_error at 0x7e232d88ed30>,
      optimizer='adam', output_activation='sigmoid', preprocessing=True,
      random_state=42, validation_size=0.1, verbose=1)

In [10]:
thres_np = autoencode.predict(df[['value']])
thres_np

1485/1485 [==============================] - 2s 1ms/step


array([0, 1, 0, ..., 0, 0, 0])

In [11]:
gtruth_np = df[['anomaly']]
gtruth_np

,anomaly
379,0
380,0
381,0
382,0
383,0
...,...
50109,0
50110,0
50111,0
50112,0


In [12]:
prec = precision_score(gtruth_np, thres_np, pos_label=1)
recall = recall_score(gtruth_np, thres_np, pos_label=1)
f1 = f1_score(gtruth_np, thres_np, pos_label=1)

In [13]:
print(f'Precsion Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precsion Score: 0.0610  Recall: 0.0572  f1_score: 0.0590
